# Implementasi Bab 6 — KNN dan Naive Bayes

Perbandingan KNN berskala dengan Gaussian Naive Bayes pada Wine, pengaruh nilai k, dan Multinomial Naive Bayes untuk teks [ML-MODUL-01, hlm. 46–47].


In [ ]:
import random
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
wine = load_wine(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=SEED, stratify=wine.target
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, weights="distance")
knn.fit(X_train_scaled, y_train)
gaussian_nb = GaussianNB().fit(X_train, y_train)

accuracy_knn = accuracy_score(y_test, knn.predict(X_test_scaled))
accuracy_nb = accuracy_score(y_test, gaussian_nb.predict(X_test))
print(f"KNN: {accuracy_knn:.3f}")
print(f"GaussianNB: {accuracy_nb:.3f}")


In [ ]:
k_values = range(1, 21)
k_cv_scores = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for k in k_values:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])
    k_cv_scores.append(cross_val_score(candidate, X_train, y_train, cv=cv).mean())
best_k = list(k_values)[int(np.argmax(k_cv_scores))]
selected_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
]).fit(X_train, y_train)
print("k terbaik menurut CV training:", best_k)
print("Akurasi test final:", round(accuracy_score(y_test, selected_knn.predict(X_test)), 3))
plt.plot(list(k_values), k_cv_scores, marker="o")
plt.xlabel("Nilai k")
plt.ylabel("Rata-rata akurasi CV")
plt.title("Pengaruh Nilai k")
plt.show()


## Latihan 1: waktu KNN vs GaussianNB

Waktu diukur secara sederhana untuk tujuan pembelajaran. Hasil dipengaruhi perangkat dan ukuran data [ML-MODUL-01, hlm. 47].


In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=SEED, stratify=cancer.target
)
candidates = {
    "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=5))]),
    "GaussianNB": GaussianNB(),
}
timing_rows = []
for name, candidate in candidates.items():
    start = time.perf_counter()
    candidate.fit(X_train_c, y_train_c)
    fit_time = time.perf_counter() - start
    start = time.perf_counter()
    prediction = candidate.predict(X_test_c)
    predict_time = time.perf_counter() - start
    timing_rows.append((name, accuracy_score(y_test_c, prediction), fit_time, predict_time))
display(pd.DataFrame(timing_rows, columns=["model", "accuracy", "fit_seconds", "predict_seconds"]))


## Latihan 2: Multinomial Naive Bayes untuk teks

CountVectorizer mengubah kalimat menjadi frekuensi kata non-negatif yang cocok untuk MultinomialNB [ML-MODUL-01, hlm. 47].


In [ ]:
texts = [
    "dapatkan diskon gratis sekarang",
    "rapat tim jam 10 pagi",
    "menangkan hadiah gratis klik disini",
    "laporan keuangan bulan ini",
    "diskon spesial hari ini",
    "jadwal kuliah minggu depan",
]
labels = [1, 0, 1, 0, 1, 0]
text_pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("model", MultinomialNB()),
])
text_pipeline.fit(texts, labels)
examples = ["hadiah gratis untuk anda", "jadwal rapat besok"]
print(list(zip(examples, text_pipeline.predict(examples))))


## Kesimpulan

KNN sensitif pada skala dan nilai k; GaussianNB memodelkan distribusi fitur kontinu; MultinomialNB cocok untuk hitungan seperti bag-of-words. Pemilihan model harus mengikuti karakteristik data.
